# 🖼️ Aiscern — Image Deepfake Detector
### Kaggle P100 | ViT-base-patch16-224 + LoRA | Full 78k samples × 10 epochs

**Platform**: Kaggle (same session as audio, ~2.3h extra)  
**GPU**: P100 16GB  
**Expected time**: ~2.3h  
**Output model**: `saghi776/aiscern-image-detector`  
**Expected accuracy**: 97% → **99%**

### Datasets used
| Dataset | Size | Type |
|---|---|---|
| CIFAKE | 60,000 | AI-generated (SDXL) vs real CIFAR-10 |
| FaceForensics++ | 18,000 | Deepfake face swaps |


In [ ]:
!pip install -q transformers==4.40.0 datasets peft==0.10.0 accelerate              evaluate scikit-learn Pillow huggingface_hub timm

import os, warnings, logging, torch
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f"✅ GPU: {gpu}")
print("✅ Dependencies installed")


In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────
HF_TOKEN        = 'YOUR_HF_TOKEN_HERE'   # ← paste your HF token

BASE_MODEL      = 'google/vit-base-patch16-224'
PUSH_REPO       = 'saghi776/aiscern-image-detector'
CHECKPOINT_DIR  = './image-checkpoints'
IMG_SIZE        = 224
SAMPLES_PER_CLASS = 39000    # all available (78,000 ÷ 2)
BATCH_SIZE      = 32         # P100 16GB handles ViT-base batch=32 fine
EPOCHS          = 10
LR              = 2e-4
WARMUP_RATIO    = 0.1
WEIGHT_DECAY    = 0.01
SEED            = 42

import os
os.environ['HF_TOKEN'] = HF_TOKEN
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Config set — {SAMPLES_PER_CLASS*2:,} samples × {EPOCHS} epochs")
print(f"   Estimated time: ~2.3h on Kaggle P100")


In [ ]:
# ── LOAD DATASETS ────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets
from PIL import Image as PILImage
import numpy as np

def normalise_label(val):
    s = str(val).upper().strip()
    if s in ('FAKE','AI','1','GENERATED','TRUE'): return 1
    if s in ('REAL','HUMAN','0','FALSE'):          return 0
    return -1

print("Loading datasets...")
all_splits = []

# 1. CIFAKE — clean binary labels, 60k samples (30k real, 30k AI-gen)
try:
    ds = load_dataset(
        'jlbaker361/cifake-real-and-ai-generated-small-images',
        split='train', token=HF_TOKEN
    )
    lbl_col = next((c for c in ds.column_names if 'label' in c.lower()), None)
    img_col = next((c for c in ds.column_names if c in ('image','img','Image')), None)
    if lbl_col and img_col:
        if img_col != 'image': ds = ds.rename_column(img_col, 'image')
        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds = ds.filter(lambda x: x['label'] != -1)
        ds = ds.select_columns(['image', 'label'])
        all_splits.append(ds)
        r = ds.filter(lambda x: x['label']==0).num_rows
        f = ds.filter(lambda x: x['label']==1).num_rows
        print(f"  ✅ CIFAKE: {len(ds):,}  (real={r:,} fake={f:,})")
except Exception as e:
    print(f"  ⚠️  CIFAKE skipped: {e}")

# 2. FaceForensics++ deepfake faces
try:
    ds = load_dataset(
        'marcelomoreno26/deepfake-detection',
        split='train', token=HF_TOKEN
    )
    lbl_col = next((c for c in ds.column_names if 'label' in c.lower()), None)
    img_col = next((c for c in ds.column_names if c in ('image','img','Image')), None)
    if lbl_col and img_col:
        if img_col != 'image': ds = ds.rename_column(img_col, 'image')
        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds = ds.filter(lambda x: x['label'] != -1)
        ds = ds.select_columns(['image', 'label'])
        all_splits.append(ds)
        r = ds.filter(lambda x: x['label']==0).num_rows
        f = ds.filter(lambda x: x['label']==1).num_rows
        print(f"  ✅ FaceForensics: {len(ds):,}  (real={r:,} fake={f:,})")
except Exception as e:
    print(f"  ⚠️  FaceForensics skipped: {e}")

# ── Balance ──────────────────────────────────────────────────────
if not all_splits:
    raise RuntimeError("No datasets loaded — check HF_TOKEN")

combined = concatenate_datasets(all_splits)
real = combined.filter(lambda x: x['label'] == 0).shuffle(SEED)
fake = combined.filter(lambda x: x['label'] == 1).shuffle(SEED)
n    = min(len(real), len(fake), SAMPLES_PER_CLASS)
balanced = concatenate_datasets([
    real.select(range(n)), fake.select(range(n))
]).shuffle(SEED)

split    = balanced.train_test_split(test_size=0.1, seed=SEED)
train_ds = split['train']
eval_ds  = split['test']
print(f"\n✅ Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}")


In [ ]:
# ── AUGMENTATION + FEATURE EXTRACTION ───────────────────────────
from transformers import ViTImageProcessor
from PIL import Image as PILImage, ImageEnhance, ImageFilter
import numpy as np, random

processor = ViTImageProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)

def augment(img, is_train=True):
    """Light augmentation — enough to improve generalisation"""
    if not isinstance(img, PILImage.Image):
        img = PILImage.fromarray(np.uint8(img)).convert('RGB')
    img = img.convert('RGB').resize((IMG_SIZE, IMG_SIZE), PILImage.LANCZOS)
    if is_train:
        # Random horizontal flip
        if random.random() > 0.5:
            img = img.transpose(PILImage.FLIP_LEFT_RIGHT)
        # Random brightness/contrast
        if random.random() > 0.5:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
        if random.random() > 0.5:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    return img

def preprocess_train(batch):
    imgs = [augment(img, is_train=True) for img in batch['image']]
    batch['pixel_values'] = processor(images=imgs, return_tensors='np')['pixel_values']
    return batch

def preprocess_eval(batch):
    imgs = [augment(img, is_train=False) for img in batch['image']]
    batch['pixel_values'] = processor(images=imgs, return_tensors='np')['pixel_values']
    return batch

print("Extracting features (with augmentation for train)...")
train_ds = train_ds.map(preprocess_train, batched=True, batch_size=64,
                         remove_columns=['image'], desc='Train')
eval_ds  = eval_ds.map(preprocess_eval,   batched=True, batch_size=64,
                         remove_columns=['image'], desc='Eval')
train_ds.set_format('torch')
eval_ds.set_format('torch')
print(f"✅ Features ready")


In [ ]:
# ── MODEL + LoRA ─────────────────────────────────────────────────
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model
import torch

model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'real', 1: 'fake'},
    label2id={'real': 0, 'fake': 1},
    ignore_mismatched_sizes=True,
    token=HF_TOKEN,
)

# LoRA on attention layers — trains ~0.5% of params
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    target_modules=['query', 'value', 'key', 'dense'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
model = model.to(device)
print(f"✅ Model on {device}")


In [ ]:
# ── TRAIN ────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate, numpy as np

acc = evaluate.load('accuracy')
f1  = evaluate.load('f1')

def compute_metrics(ep):
    preds = np.argmax(ep.predictions, axis=-1)
    return {
        'accuracy': acc.compute(predictions=preds, references=ep.label_ids)['accuracy'],
        'f1':       f1.compute(predictions=preds, references=ep.label_ids, average='binary')['f1'],
    }

training_args = TrainingArguments(
    output_dir              = CHECKPOINT_DIR,
    num_train_epochs        = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate           = LR,
    warmup_ratio            = WARMUP_RATIO,
    weight_decay            = WEIGHT_DECAY,
    eval_strategy           = 'epoch',
    save_strategy           = 'epoch',
    save_total_limit        = 3,
    load_best_model_at_end  = True,
    metric_for_best_model   = 'f1',
    greater_is_better       = True,
    push_to_hub             = True,
    hub_model_id            = PUSH_REPO,
    hub_token               = HF_TOKEN,
    hub_strategy            = 'every_save',
    fp16                    = (device == 'cuda'),
    dataloader_num_workers  = 4,
    report_to               = 'none',
    logging_steps           = 100,
    seed                    = SEED,
    resume_from_checkpoint  = True,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Starting training: {len(train_ds):,} samples × {EPOCHS} epochs")
trainer.train()
print("\n✅ Training complete!")


In [ ]:
# ── EVALUATE + PUSH ──────────────────────────────────────────────
results = trainer.evaluate()

print("\n" + "═"*50)
print("FINAL RESULTS")
print("═"*50)
print(f"  Accuracy: {results['eval_accuracy']:.4f}  ({results['eval_accuracy']*100:.2f}%)")
print(f"  F1 Score: {results['eval_f1']:.4f}")
print("═"*50)

trainer.push_to_hub(commit_message=f"Image deepfake detector — acc={results['eval_accuracy']:.4f}")
processor.push_to_hub(PUSH_REPO, token=HF_TOKEN)

from huggingface_hub import HfApi
card = f"""---
license: apache-2.0
tags:
- image-classification
- deepfake-detection
- vit
- lora
datasets:
- jlbaker361/cifake-real-and-ai-generated-small-images
- marcelomoreno26/deepfake-detection
metrics:
- accuracy
---

# Aiscern Image Deepfake Detector

Fine-tuned `google/vit-base-patch16-224` with LoRA for AI image detection.

**Accuracy**: {results['eval_accuracy']*100:.2f}%  
**Training data**: {len(train_ds):,} samples (CIFAKE + FaceForensics++)

## Usage
```python
from transformers import pipeline
detector = pipeline('image-classification', model='{PUSH_REPO}')
result = detector('image.jpg')
# [{{'label': 'fake', 'score': 0.99}}]
```
"""

HfApi().upload_file(
    path_or_fileobj=card.encode(), path_in_repo='README.md',
    repo_id=PUSH_REPO, token=HF_TOKEN,
)
print(f"\n✅ Model live: https://huggingface.co/{PUSH_REPO}")
